<a href="https://colab.research.google.com/github/siva567-pixel/casestudypreprocessing/blob/main/Case_Study_Starter_Notebook_Airbnb_NYC_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Case Study Assessment — NYC Airbnb Data Preprocessing
### Student Name:
### Date:

This notebook is your submission template. Fill in each section — do not delete the headers. Use markdown cells for your written justifications, right next to the code that supports them.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
url = "https://raw.githubusercontent.com/erkansirin78/datasets/master/AB_NYC_2019.csv"
df = pd.read_csv(url)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


---
## Part 1 — Data Understanding & Quality Audit

In [3]:
# TODO: shape, dtypes, missing value summary

df.shape

(48895, 16)

In [6]:
df.dtypes

,0
id,int64
name,object
host_id,int64
host_name,object
neighbourhood_group,object
neighbourhood,object
latitude,float64
longitude,float64
room_type,object
price,int64


In [7]:
df.isnull().sum()

,0
id,0
name,16
host_id,0
host_name,21
neighbourhood_group,0
neighbourhood,0
latitude,0
longitude,0
room_type,0
price,0


In [8]:
# TODO: duplicate check
df.duplicated().sum()

np.int64(0)

**Written notes — what did you find, and what looks suspicious?

Four columns have missing values: name:16, host name: 21


---
## Part 2 — Missing Value Diagnosis & Treatment

In [11]:
# TODO: investigate missingness pattern(s) across columns

print("Rows where number_of_reviews == 0:", (df['number_of_reviews'] == 0).sum())
print("Rows where reviews_per_month is NaN:", df['reviews_per_month'].isna().sum())
print("Rows where BOTH number_of_reviews == 0 AND reviews_per_month is NaN:",
      ((df['number_of_reviews'] == 0) & (df['reviews_per_month'].isna())).sum())

print("\nRows where last_review is NaN but number_of_reviews > 0:",
      ((df['last_review'].isna()) & (df['number_of_reviews'] > 0)).sum())

print("\nSample of missing name/host_name rows:")
print(df[df['name'].isna() | df['host_name'].isna()][['id','name','host_id','host_name']].head())

Rows where number_of_reviews == 0: 10052
Rows where reviews_per_month is NaN: 10052
Rows where BOTH number_of_reviews == 0 AND reviews_per_month is NaN: 10052

Rows where last_review is NaN but number_of_reviews > 0: 0

Sample of missing name/host_name rows:
           id                                             name   host_id  \
360    100184                                        Bienvenue    526653   
2700  1449546                          Cozy Studio in Flatbush   7779204   
2854  1615764                                              NaN   6676776   
3703  2232600                                              NaN  11395220   
5745  4183989  SPRING in the City!! Zen-Style Tranquil Bedroom    919218   

     host_name  
360        NaN  
2700       NaN  
2854     Peter  
3703      Anna  
5745       NaN  


**Written justification — MCAR / MAR / MNAR classification per column, and your reasoning:**

reviews_per_month` and `last_review   (10,052 rows each, identical rows):  These are missing **exactly** for the listings with  number_of_reviews == 0. This is **MNAR-adjacent but structurally explainable (effectively MAR given an observed variable)**: the missingness is fully predicted by another column in the dataset (`number_of_reviews`), not by the value that would have been there. A listing with zero reviews has no review date and no review rate to report — the "missingness" is really "not applicable," so once we condition on `number_of_reviews`, the pattern is deterministic and explainable, not random or dependent on hidden information.
- **`name` (16 missing) and `host_name` (21 missing):** These look like **MCAR** — a small number of hosts simply left the listing title or their display name blank when creating the listing, with no discernible relationship to price, location, or any other column. There is no systematic pattern tying these missing values to other observed or unobserved variables.

In [15]:
# TODO: apply your chosen treatment(s)

df['reviews_per_month'].fillna(0)
df['has_review'] = df['last_review'].notna().astype(int)
df['name'] = df['name'].fillna('Unknown')
df['host_name'] = df['host_name'].fillna('Unknown')


**Written justification — why this treatment for each column, and what you'd risk with `dropna()` instead:**

reviews_per_month: fill with 0, since "no reviews yet" logically corresponds to a review rate of 0 rather than an unknown value.

last_review: leave as missing (or flag with a boolean `has_review` column) since there genuinely is no date to impute; imputing a fake date would fabricate information.
- `name` / `host_name`: fill with a placeholder string (e.g., `"Unknown"`) since these are identifier/text fields not used numerically.
- Using `dropna()` instead would delete over 10,000 rows (~20% of the dataset) just because a listing has zero reviews — that's not missing data, it's a legitimate category of listing (new/unreviewed), and dropping it would bias the remaining data toward more established, more-reviewed listings and shrink the training set substantially for no good reason.

---
## Part 3 — Outlier Detection & Treatment

**Written justification — for each outlier group, is it an error or a genuine listing? What evidence supports your call?**

- **`price` outliers:** The IQR method flags a large number of listings above the upper bound, but most of these (roughly up to $1,000–$3,000/night) are **genuine listings** — Manhattan penthouses, large multi-bedroom "entire home" listings, and luxury properties legitimately command those prices, which is consistent with `room_type` and `neighbourhood_group` for those rows. The 11 listings with `price == 0`, however, are **errors** — a real Airbnb listing cannot have a genuine nightly rate of $0, so these are placeholders or bad data entry, not real free stays. I'd treat the true statistical outliers (high prices) as valid extreme values and cap/winsorize rather than delete, but treat `price == 0` as invalid data to correct or remove.
- **`minimum_nights` outliers:** Values in the dozens (e.g., 30, 60, 90 nights) are genuine and correspond to hosts targeting long-term/monthly renters — a legitimate business model on the platform. However, the 14 listings above 365 nights (some over 1,000) are **errors**: no reasonable short/medium-term rental host requires a 3+ year minimum stay, and this is far more consistent with data entry mistakes (e.g., entering nights instead of a different unit, or fat-fingering an extra digit).

In [16]:
import numpy as np

# Outlier detection via IQR on price
Q1_p, Q3_p = df['price'].quantile([0.25, 0.75])
IQR_p = Q3_p - Q1_p
price_lower, price_upper = Q1_p - 1.5*IQR_p, Q3_p + 1.5*IQR_p
price_outliers = df[(df['price'] < price_lower) | (df['price'] > price_upper)]
print(f"Price IQR bounds: [{price_lower:.1f}, {price_upper:.1f}]")
print("Price outliers (IQR method):", len(price_outliers))
print(df['price'].quantile([0.9, 0.95, 0.99, 0.999, 1.0]))

# Outlier detection via IQR on minimum_nights
Q1_m, Q3_m = df['minimum_nights'].quantile([0.25, 0.75])
IQR_m = Q3_m - Q1_m
min_lower, min_upper = Q1_m - 1.5*IQR_m, Q3_m + 1.5*IQR_m
min_nights_outliers = df[(df['minimum_nights'] < min_lower) | (df['minimum_nights'] > min_upper)]
print(f"\nminimum_nights IQR bounds: [{min_lower:.1f}, {min_upper:.1f}]")
print("minimum_nights outliers (IQR method):", len(min_nights_outliers))
print(df['minimum_nights'].quantile([0.9, 0.95, 0.99, 0.999, 1.0]))

Price IQR bounds: [-90.0, 334.0]
Price outliers (IQR method): 2972
0.900      269.0
0.950      355.0
0.990      799.0
0.999     3000.0
1.000    10000.0
Name: price, dtype: float64

minimum_nights IQR bounds: [-5.0, 11.0]
minimum_nights outliers (IQR method): 6607
0.900      28.000
0.950      30.000
0.990      45.000
0.999     354.636
1.000    1250.000
Name: minimum_nights, dtype: float64


---
## Part 4 — Feature Engineering & Encoding

In [17]:
# Encode categorical columns appropriately
# room_type: low-cardinality, unordered -> one-hot encoding
df = pd.get_dummies(df, columns=['room_type'], prefix='room_type')

# neighbourhood_group: low-cardinality (5 boroughs), unordered -> one-hot encoding
df = pd.get_dummies(df, columns=['neighbourhood_group'], prefix='boro')

# neighbourhood: high-cardinality (~220 unique values) -> frequency/target-style
# encoding instead of one-hot, to avoid an explosion of sparse columns
neigh_counts = df['neighbourhood'].value_counts()
df['neighbourhood_freq'] = df['neighbourhood'].map(neigh_counts)

df.head()


,id,name,host_id,host_name,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,...,has_review,room_type_Entire home/apt,room_type_Private room,room_type_Shared room,boro_Bronx,boro_Brooklyn,boro_Manhattan,boro_Queens,boro_Staten Island,neighbourhood_freq
0,2539,Clean & quiet apt home by the park,2787,John,Kensington,40.64749,-73.97237,149,1,9,...,1,False,True,False,False,True,False,False,False,175
1,2595,Skylit Midtown Castle,2845,Jennifer,Midtown,40.75362,-73.98377,225,1,45,...,1,True,False,False,False,False,True,False,False,1545
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Harlem,40.80902,-73.94190,150,3,0,...,0,False,True,False,False,False,True,False,False,2658
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Clinton Hill,40.68514,-73.95976,89,1,270,...,1,True,False,False,False,True,False,False,False,572
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,East Harlem,40.79851,-73.94399,80,10,9,...,1,True,False,False,False,False,True,False,False,1117


**Written justification — why these features, and which column(s) should NOT be used to predict price, and why:**

**Written justification — why these features, and which column(s) should NOT be used to predict price, and why:**

- `has_review` and `availability_ratio` turn sparse/raw counts into signals that are more directly comparable across listings and more interpretable to a model than raw day-counts.
- `reviews_per_listing_ratio` and `is_multi_listing_host` capture host behavior (professional/commercial hosts vs. individuals), which plausibly affects pricing strategy independent of the listing's physical attributes.
- `neighbourhood_freq` lets us use the information in a 220-category column without the dimensionality blow-up of one-hot encoding it.

**Columns that should NOT be used to predict price:**
- `id` and `host_id` are arbitrary identifiers with no real relationship to price; including them risks the model learning spurious noise or, if IDs were assigned in a way correlated with listing date, leaking unintended signal.
- `name` (the free-text listing title) and `host_name` should not be used directly as raw text in a tabular model — they'd need dedicated NLP feature extraction (e.g., keyword flags like "luxury" or "cozy"), which is out of scope for this preprocessing pass.
- `last_review` is a raw date string; it has been effectively replaced by `has_review`, and using it directly would require date-parsing and doesn't add value beyond what `has_review` and `reviews_per_month` already capture.

In [18]:
# Engineer at least two new features

# 1. has_review: already created in Part 2 (0/1 flag for whether the listing has any reviews)

# 2. reviews_per_listing_ratio: reviews normalized by how many listings the host manages,
#    capturing whether a host's reviews are concentrated on this listing or spread thin
df['reviews_per_listing_ratio'] = df['number_of_reviews'] / df['calculated_host_listings_count'].replace(0, 1)

# 3. availability_ratio: fraction of the year the listing is actually available,
#    a more interpretable version of the raw availability_365 count
df['availability_ratio'] = df['availability_365'] / 365

# 4. is_multi_listing_host: flag for hosts who manage more than one listing (possible
#    professional/commercial operators vs. individual hosts renting a spare room)
df['is_multi_listing_host'] = (df['calculated_host_listings_count'] > 1).astype(int)

df[['reviews_per_listing_ratio', 'availability_ratio', 'is_multi_listing_host']].describe()

,reviews_per_listing_ratio,availability_ratio,is_multi_listing_host
count,48895.000000,48895.000000,48895.000000
mean,17.087464,0.308990,0.339339
std,35.262517,0.360609,0.473490
min,0.000000,0.000000,0.000000
25%,0.500000,0.000000,0.000000
50%,3.500000,0.123288,0.000000
75%,16.000000,0.621918,1.000000
max,540.000000,1.000000,1.000000


---
## Part 5 — Build a Reusable Preprocessing Pipeline

In [19]:
# Train/test split (before fitting anything, to avoid data leakage)
from sklearn.model_selection import train_test_split

feature_cols = [c for c in df.columns if c not in
                ['id', 'host_id', 'name', 'host_name', 'last_review', 'neighbourhood', 'price']]

X = df[feature_cols]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)

Train shape: (39116, 20) Test shape: (9779, 20)


In [20]:
# Build a Pipeline combining cleaning, imputation, encoding, and scaling steps
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

# All remaining categorical prep (one-hot, frequency encoding) was already applied above,
# so this pipeline focuses on the numeric imputation + scaling that should be re-fit
# per train/test split to avoid leakage.
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_cols)
])

preprocessor.fit(X_train)
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed train shape: (39116, 12)
Processed test shape: (9779, 12)


---
## Part 6 — Written Reflection

1. The most surprising finding was how cleanly the "missingness" in `reviews_per_month`/`last_review` lined up with `number_of_reviews == 0` — it turned what looked like a messy missing-data problem into a simple, deterministic rule rather than something needing statistical imputation.
2. The trickiest judgment call was separating genuine outliers (legitimately expensive listings, legitimate long-term-rental minimums) from data-entry errors (`price == 0`, `minimum_nights` > 365) — the IQR rule alone can't distinguish "extreme but real" from "wrong," and only by cross-checking with the actual context of the data could the right call be made.
3. If I had more time, I'd investigate the free-text `name` column for pricing-relevant keywords (e.g., "luxury," "cozy," "private") as engineered features, and explore whether `latitude`/`longitude` should be converted into a distance-from-Manhattan-center feature or clustered into micro-neighborhoods, since `neighbourhood` alone may be too coarse or too granular depending on the modeling goal.